# LLM · Lecture 9 — Structured retrieval with **SQL** and the **star schema**

Query construction is: an LLM turns a natural-language question into a **structured query** it can actually run. When the source is a database, that structured query is **SQL**. This notebook teaches the three SQL moves you need — **filter (`WHERE`)**, **aggregate (`GROUP BY`)**, **join** — then the **star schema** that makes analytics (and text-to-SQL) easy: nearly every business question becomes *join → filter → aggregate*.

We use a small but realistic **retail** dataset. Pure `sqlite3` + `pandas` — both preinstalled in Colab, nothing to install. Run each cell top to bottom.

> **Why this matters for LLM apps.** A model that writes SQL over a well-shaped warehouse can answer questions plain semantic search never could — *"revenue by category last quarter in the North region"*. The star schema is what makes the model's job reliable.

## 0. Setup — build a tiny retail data warehouse

In [ ]:
import sqlite3, textwrap, re, os
import numpy as np, pandas as pd
pd.set_option('display.max_rows', 20)
rng = np.random.default_rng(7)

# ---- dimension tables: the descriptive 'who / what / when / where' ----
products = pd.DataFrame({
    'product_id': range(1, 13),
    'product': ['UltraBook 14','UltraBook 16','Phone X','Phone X Mini','Buds Pro','Buds Lite',
                'Over-Ear 900','USB-C Charger','Laptop Sleeve','Phone Case','4K Monitor','Webcam'],
    'category': ['Laptops','Laptops','Phones','Phones','Audio','Audio',
                 'Audio','Accessories','Accessories','Accessories','Monitors','Accessories'],
    'brand': ['Acme','Acme','Nova','Nova','Nova','Nova','Acme','Volt','Volt','Nova','Acme','Volt'],
    'unit_price': [1200,1600,900,650,180,90,240,25,20,15,320,60],
})
regions = ['North','South','Central','East']
segments = ['Consumer','Business']
customers = pd.DataFrame({
    'customer_id': range(1, 201),
    'customer': [f'Cust{i:03d}' for i in range(1, 201)],
    'region': rng.choice(regions, 200),
    'segment': rng.choice(segments, 200, p=[0.7, 0.3]),
})
dates = pd.date_range('2024-01-01', '2025-12-31', freq='D')
dim_date = pd.DataFrame({
    'date_id': range(1, len(dates)+1), 'date': dates.astype(str),
    'year': dates.year, 'quarter': dates.quarter, 'month': dates.month,
    'weekday': dates.day_name(),
})

# ---- fact table: one row per sale (the measurements: units, revenue) ----
N = 6000
pid = rng.integers(1, 13, N)
price = products.set_index('product_id').loc[pid, 'unit_price'].to_numpy()
units = rng.integers(1, 6, N)
discount = np.round(rng.choice([0,0,0,0.1,0.15,0.2], N), 2)
fact = pd.DataFrame({
    'sale_id': range(1, N+1),
    'date_id': rng.integers(1, len(dates)+1, N),
    'product_id': pid,
    'customer_id': rng.integers(1, 201, N),
    'units': units,
    'discount': discount,
    'revenue': np.round(units * price * (1 - discount), 2),
})

con = sqlite3.connect(':memory:')
products.to_sql('dim_product', con, index=False)
customers.to_sql('dim_customer', con, index=False)
dim_date.to_sql('dim_date', con, index=False)
fact.to_sql('fact_sales', con, index=False)

def q(sql):
    """Run SQL and return the result as a DataFrame (renders as a table)."""
    return pd.read_sql_query(textwrap.dedent(sql), con)

print('tables:', [r[0] for r in con.execute("select name from sqlite_master where type='table'")])
print('fact_sales rows:', len(fact))

## 1. Look at the data first

Four tables. The **fact** table (`fact_sales`) has one row per sale and holds the *numbers* (units, revenue) plus **id** links; the **dimension** tables describe those ids in words.

In [ ]:
q('SELECT * FROM fact_sales LIMIT 5')

In [ ]:
q('SELECT * FROM dim_product')

In [ ]:
q('SELECT * FROM dim_customer LIMIT 5')

## 2. Filtering rows — `WHERE`

`WHERE` keeps only the rows that match a condition. This is the SQL version of the **metadata filter** from the lecture — the *hard constraints* in a question.

In [ ]:
# 'Show sales of 3 or more units'
q('SELECT sale_id, product_id, units, revenue FROM fact_sales WHERE units >= 3 LIMIT 5')

In [ ]:
# comparison, ranges, sets, text — the everyday operators
q("SELECT product, category, unit_price FROM dim_product WHERE unit_price BETWEEN 100 AND 700")

In [ ]:
# IN (a set), LIKE (text pattern), AND / OR to combine
q("""
    SELECT product, category, brand, unit_price FROM dim_product
    WHERE category IN ('Phones','Audio')
      AND (brand = 'Nova' OR unit_price < 100)
""")

Common `WHERE` operators: `=  <>  <  >  <=  >=`, `BETWEEN a AND b`, `IN (…)`, `LIKE '%pattern%'`, `IS NULL`, combined with `AND` / `OR` / `NOT`.

## 3. Aggregating — `GROUP BY`

Aggregation collapses many rows into a summary: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`. `GROUP BY` computes one summary *per group*. This answers the *"how much / how many"* questions.

In [ ]:
# one number over the whole table
q('SELECT COUNT(*) AS sales, ROUND(SUM(revenue),0) AS total_revenue, ROUND(AVG(revenue),1) AS avg_sale FROM fact_sales')

In [ ]:
# a number PER GROUP: revenue per product_id, biggest first
q("""
    SELECT product_id, COUNT(*) AS sales, ROUND(SUM(revenue),0) AS revenue
    FROM fact_sales
    GROUP BY product_id
    ORDER BY revenue DESC
    LIMIT 5
""")

In [ ]:
# HAVING filters the GROUPS (WHERE filters rows, HAVING filters aggregates)
q("""
    SELECT product_id, ROUND(SUM(revenue),0) AS revenue
    FROM fact_sales GROUP BY product_id
    HAVING SUM(revenue) > 400000
    ORDER BY revenue DESC
""")

> `WHERE` filters **rows before** grouping; `HAVING` filters **groups after**. `ORDER BY … DESC` sorts; `LIMIT` caps the rows — together they give you "top-N".

## 4. Joining tables — `JOIN`

`fact_sales` only stores `product_id`, not the category. To group by **category** we must **join** the fact to `dim_product` on the shared key, then we can filter and aggregate on the words, not the ids.

In [ ]:
# revenue per CATEGORY = join fact -> product, then group
q("""
    SELECT p.category, COUNT(*) AS sales, ROUND(SUM(f.revenue),0) AS revenue
    FROM fact_sales f
    JOIN dim_product p ON p.product_id = f.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""")

`JOIN … ON key` matches each fact row to its dimension row. A **star schema** is designed so this join is always *fact → dimension on an id* — one hop, no chains.

> **Portability note.** SQLite lets you `GROUP BY` an id and still select its descriptive columns; stricter databases (e.g. Postgres) want every non-aggregated column listed in the `GROUP BY`. Group by the descriptive column (or list them all) to stay portable.

## 5. The data warehouse & the star schema

A **data warehouse** is a database shaped for *analytics* (reading and summarizing history), not for running the app. The dominant design is the **star schema**:

```
      dim_date          dim_customer
          \                 /
           \               /
            +--  fact_sales  --+        <- the FACT: one row per event,
           /               \             numeric measures + id keys
          /                 \
     dim_product         (more dims…)   <- DIMENSIONS: who / what / when / where
```

- The **fact** table is long and numeric: the *measurements* (units, revenue) plus a foreign **key** to each dimension. Millions of rows.
- Each **dimension** is short and descriptive: the *context* you slice by (category, region, month, segment).

**Why it's built this way — every analytics question becomes the same three moves:**

1. **JOIN** the fact to the dimension(s) you want to slice by (always one hop, on a key),
2. **WHERE** to keep the slice you care about,
3. **GROUP BY** + an aggregate to summarize.

That is the whole game. Here it is in one query — *"revenue by category, per quarter, for Business customers in the North, in 2025"*:

In [ ]:
q("""
    SELECT d.quarter, p.category, ROUND(SUM(f.revenue),0) AS revenue
    FROM fact_sales f
    JOIN dim_product  p ON p.product_id  = f.product_id
    JOIN dim_customer c ON c.customer_id = f.customer_id
    JOIN dim_date     d ON d.date_id     = f.date_id
    WHERE d.year = 2025 AND c.region = 'North' AND c.segment = 'Business'
    GROUP BY d.quarter, p.category
    ORDER BY d.quarter, revenue DESC
""")

Same three moves, different business questions — this is what analytics *is*. A few real use cases (read each as join → filter → aggregate):

In [ ]:
# USE CASE 1 — Top 5 customers by spend in the East region
q("""
    SELECT c.customer, c.segment, ROUND(SUM(f.revenue),0) AS spend
    FROM fact_sales f JOIN dim_customer c ON c.customer_id = f.customer_id
    WHERE c.region = 'East'
    GROUP BY c.customer_id
    ORDER BY spend DESC
    LIMIT 5
""")

In [ ]:
# USE CASE 2 — Monthly revenue trend for the Audio category in 2025
q("""
    SELECT d.month, ROUND(SUM(f.revenue),0) AS revenue
    FROM fact_sales f
    JOIN dim_product p ON p.product_id = f.product_id
    JOIN dim_date    d ON d.date_id    = f.date_id
    WHERE p.category = 'Audio' AND d.year = 2025
    GROUP BY d.month ORDER BY d.month
""")

In [ ]:
# USE CASE 3 — Does discounting move volume? avg units & revenue by discount level
q("""
    SELECT f.discount, COUNT(*) AS sales, ROUND(AVG(f.units),2) AS avg_units,
           ROUND(SUM(f.revenue),0) AS revenue
    FROM fact_sales f
    GROUP BY f.discount ORDER BY f.discount
""")

In [ ]:
# USE CASE 4 — Best-selling brand in each region (revenue), Consumer segment
q("""
    SELECT c.region, p.brand, ROUND(SUM(f.revenue),0) AS revenue
    FROM fact_sales f
    JOIN dim_product  p ON p.product_id  = f.product_id
    JOIN dim_customer c ON c.customer_id = f.customer_id
    WHERE c.segment = 'Consumer'
    GROUP BY c.region, p.brand
    ORDER BY c.region, revenue DESC
""")

Notice the shape never really changes: pick the dimensions to **join**, the slice to **`WHERE`**, and the measure to **`GROUP BY` + aggregate**. That predictability is the point of the star schema — and it's exactly what makes an LLM able to write these queries reliably.

## 6. From a natural-language question to SQL — *query construction*

Now the LLM's job (the heart of Lecture 9): read an English question and **emit the SQL** that answers it over this star schema. Because the schema is a star, the model only has to decide three things — which **dimensions to join**, what to **filter**, what to **group/aggregate** — the joins are always `fact → dim` on a key.

Here are realistic **question → SQL** pairs. Read the English, then the query it becomes:

In [ ]:
examples = {
  'How many Phones did we sell in 2024?':
    """SELECT SUM(f.units) AS units FROM fact_sales f
       JOIN dim_product p ON p.product_id=f.product_id
       JOIN dim_date d ON d.date_id=f.date_id
       WHERE p.category='Phones' AND d.year=2024""",
  'Total revenue by region, best first':
    """SELECT c.region, ROUND(SUM(f.revenue),0) AS revenue FROM fact_sales f
       JOIN dim_customer c ON c.customer_id=f.customer_id
       GROUP BY c.region ORDER BY revenue DESC""",
  'Which laptop had the most revenue in Q4 2025?':
    """SELECT p.product, ROUND(SUM(f.revenue),0) AS revenue FROM fact_sales f
       JOIN dim_product p ON p.product_id=f.product_id
       JOIN dim_date d ON d.date_id=f.date_id
       WHERE p.category='Laptops' AND d.year=2025 AND d.quarter=4
       GROUP BY p.product_id ORDER BY revenue DESC LIMIT 1""",
}
for question, sql in examples.items():
    print('Q:', question)
    print(q(sql).to_string(index=False), '\n')

### The prompt an LLM would use

You give the model the **schema** (tables + columns) and ask for SQL only. A minimal template:

In [ ]:
SCHEMA = '''
fact_sales(sale_id, date_id, product_id, customer_id, units, discount, revenue)
dim_product(product_id, product, category, brand, unit_price)
dim_customer(customer_id, customer, region, segment)
dim_date(date_id, date, year, quarter, month, weekday)
Joins: fact_sales.product_id=dim_product.product_id, .customer_id=dim_customer.customer_id, .date_id=dim_date.date_id
'''
SYSTEM = ('You translate a question into ONE read-only SQLite SELECT over this schema. '
          'Return SQL only, no prose.\n' + SCHEMA)

print(SYSTEM)

### Make it live — let the model actually write the SQL

The class LLM is an **OpenAI-compatible** proxy, so we point the standard `openai` client at it. Get your key from the course portal (**Profile → API key**), paste it below, and run — the model reads the question and emits the SQL, which we **validate** (single read-only `SELECT`) and then run. No key? The cell skips the call and the hand-written examples above still stand, so the notebook runs end-to-end either way.

In [ ]:
os.environ.setdefault('OPENAI_BASE_URL', 'https://llm.nat-d.uk/v1')   # the class proxy
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # <-- paste your key from the portal (Profile -> API key)
MODEL = 'gemma-4-E4B-it'

def ask_sql(question, model=MODEL):
    from openai import OpenAI                 # preinstalled in Colab
    client = OpenAI()                         # reads OPENAI_API_KEY + OPENAI_BASE_URL
    r = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{'role': 'system', 'content': SYSTEM},
                  {'role': 'user',   'content': question}])
    sql = r.choices[0].message.content.strip()
    return re.sub(r'^```(?:sql)?|```$', '', sql, flags=re.I | re.M).strip()  # strip a code fence

def is_safe(sql):
    s = sql.strip().rstrip(';')
    return (s.lower().startswith('select') and ';' not in s
            and not re.search(r'\b(insert|update|delete|drop|alter|create|attach|pragma)\b', s, re.I))

question = 'What was the total revenue for the Audio category in 2025?'
if os.environ.get('OPENAI_API_KEY'):
    sql = ask_sql(question)
    print('the model wrote:\n', sql, '\n')
    if is_safe(sql):
        display(q(sql))                       # validated -> run it and show the answer
    else:
        print('refused: not a single read-only SELECT (fall back to plain search).')
else:
    print('No OPENAI_API_KEY set - skipping the live call.')
    print('Get a key from the course portal (Profile -> API key), set OPENAI_API_KEY above, re-run.')

### Guardrails — the model's SQL is untrusted input

Same rule as the rest of Lecture 9: never trust the model's output blindly. In production you:

- run it on a **read-only** connection (no `INSERT/UPDATE/DELETE/DROP`),
- **allow-list** the tables/columns and reject anything else,
- add a `LIMIT`, and pass any literal values as **parameters**, never string-spliced,
- validate it parses and only then execute — otherwise fall back to plain search.

A structured, star-shaped schema shrinks the attack surface *and* the space of queries the model can get wrong.

In [ ]:
# Parameters: values are BOUND, never spliced into the SQL string -> injection-proof
safe_sql = 'SELECT COUNT(*) AS n FROM fact_sales WHERE revenue > ?'
print('rows with revenue > 1000:', pd.read_sql_query(safe_sql, con, params=(1000,)).iloc[0, 0])
# a malicious value lands as a bound parameter (a string), never as executable SQL:
print('injection attempt as a param -> just data:',
      pd.read_sql_query(safe_sql, con, params=('0; DROP TABLE fact_sales',)).iloc[0, 0])

# Read-only: put the connection in query-only mode; any write is then refused.
con.execute('PRAGMA query_only = ON')
try:
    con.execute('DROP TABLE fact_sales')
except sqlite3.OperationalError as e:
    print('write refused on a read-only connection:', e)
con.execute('PRAGMA query_only = OFF')   # restore for the rest of the notebook

## Your turn

Write the SQL (and imagine the English question a user would ask):

1. **Filter + aggregate.** Total revenue from the `Business` segment in 2025.
2. **Top-N.** The 3 products with the highest average discount.
3. **Two joins.** Revenue by `category` for customers in the `South`, per `year`.
4. **HAVING.** Regions whose total revenue exceeds 1,000,000.
5. **Design.** Add a `dim_store(store_id, store, region, channel)` dimension and a `store_id` on `fact_sales`; then write "revenue by channel per quarter". (What changed in your query? Just one more join.)
6. **Query construction.** Write the `SYSTEM` prompt you'd send an LLM, then hand-write the SQL for *"average order value by segment last quarter"* — the answer the model should produce.

## Recap
- **`WHERE`** filters rows, **`GROUP BY` + aggregate** summarizes, **`JOIN`** brings in the descriptive columns — the three moves of analytics.
- A **star schema** (one fact + surrounding dimensions) makes *every* analytics question the same shape: **join → filter → aggregate**, joins always `fact → dim` on a key.
- That structure is what lets an LLM do **query construction** reliably: turn a natural-language question into runnable SQL — validated, read-only, parameterised.